# Обучение модели ALS (Alternating Least Squares)

## Описание
Данный ноутбук выполняет:
1. Загрузку предобработанной матрицы user-item
2. Обучение модели ALS с BPR-оптимизацией через библиотеку `implicit`
3. Извлечение эмбеддингов пользователей и товаров
4. Сохранение эмбеддингов и параметров обучения

## Модель ALS

```python
class AlternatingLeastSquares:
    """
    Матричная факторизация с чередующейся оптимизацией методом наименьших квадратов.
    
    Attributes:
        factors (int): размерность латентного пространства (k=30)
        regularization (float): коэффициент L2-регуляризации (0.01)
        iterations (int): число итераций обучения (8)
        user_factors (np.ndarray): матрица эмбеддингов пользователей [n_users x k]
        item_factors (np.ndarray): матрица эмбеддингов товаров [n_items x k]
    
    Methods:
        fit(R_items_users): обучает модель на матрице (items x users)
    """

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
%cd /content/drive/MyDrive/VKR_TECD/

/content/drive/MyDrive/VKR_TECD


In [ ]:
!pip install -q implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import numpy as np
from scipy.sparse import load_npz, csr_matrix
import implicit
import json
import time

print("ШАГ 2: ОБУЧЕНИЕ ALS (матричная факторизация с BPR)")

# Загрузка матрицы
R = load_npz("output/train_matrix.npz")
print(f"Матрица: {R.shape[0]:,} x {R.shape[1]:,}")

# implicit работает с матрицей (items x users)
R_items_users = R.T.tocsr()

factors = 30          # вместо 50
regularization = 0.01
iterations = 8        # вместо 15

print(f"factors = {factors}")
print(f"regularization = {regularization}")
print(f"iterations = {iterations}")

print("Обучение модели ALS...")
start_time = time.time()

model = implicit.als.AlternatingLeastSquares(
    factors=factors,
    regularization=regularization,
    iterations=iterations,
    calculate_training_loss=True,
    random_state=42
)

model.fit(R_items_users)

train_time = time.time() - start_time
print(f"Обучение завершено за {train_time:.2f} секунд")

# Получение эмбеддингов
user_factors = model.user_factors   # items x factors? Нет, implicit: user_factors = items?
# В implicit: model.user_factors - это эмбеддинги для строк матрицы (items)
# model.item_factors - это эмбеддинги для столбцов матрицы (users)
item_embeddings = model.user_factors   # items x factors
user_embeddings = model.item_factors   # users x factors

print(f"Матрица пользователей: {user_embeddings.shape}")
print(f"Матрица объектов: {item_embeddings.shape}")

# Сохранение
np.save("output/user_factors_als.npy", user_embeddings)
np.save("output/item_factors_als.npy", item_embeddings)

params = {
    'model': 'ALS (implicit)',
    'factors': factors,
    'regularization': regularization,
    'iterations': iterations,
    'train_time_sec': train_time
}
with open("output/als_params.json", "w") as f:
    json.dump(params, f, indent=2)

print("Эмбеддинги ALS сохранены в output/")

ШАГ 2: ОБУЧЕНИЕ ALS (матричная факторизация с BPR)
Матрица: 1,382,085 x 760,794
factors = 30
regularization = 0.01
iterations = 8
Обучение модели ALS...


/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/8 [00:00<?, ?it/s]

Обучение завершено за 713.70 секунд
Матрица пользователей: (1382085, 30)
Матрица объектов: (760794, 30)
Эмбеддинги ALS сохранены в output/
